### Load in the Model (Gemma 4 Edge / Small Models)


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Latest 2026 Small/Edge Models: 'google/gemma-4-e2b-it', 'microsoft/Phi-4-mini', 'Qwen/Qwen3-3B-Instruct'
MODEL_ID = "google/gemma-4-e2b-it"  # Gemma 4 Edge 2B

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    # Use bfloat16 to match Gemma/Qwen's native precision and avoid VRAM bloat
    torch_dtype=torch.bfloat16, 
    device_map="cuda"
)

# Dynamically find the exact middle layer (e.g. 14 for 7B, 21 for 9B, 16 for 8B)
MIDDLE_LAYER = getattr(model.config, 'text_config', model.config).num_hidden_layers // 2
print(f"Device: {model.device} | Middle Layer: {MIDDLE_LAYER}")

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

c:\Users\anael\miniconda3\envs\ml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 1951/1951 [00:09<00:00, 206.12it/s]


Device: cuda:0 | Middle Layer: 17
I am Gemma 4, a Large Language Model developed by Google DeepMind. I am an open weights model.<turn|>


### Generating Activations

In [2]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("../../datasets/plain_dataset"),
    Path("../datasets/plain_dataset"),
    Path("datasets/plain_dataset"),
    Path("Algoverse_Truth_Directions_Research/datasets/plain_dataset")
]
DATA_DIR = next((p for p in candidate_paths if p.exists() and (p / "F0_train.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Dataset directory not found.")

# Load in the Factual Datasets
F0_train, F0_test = pd.read_csv(DATA_DIR / "F0_train.csv")[["statement", "label"]], pd.read_csv(DATA_DIR / "F0_test.csv")[["statement", "label"]]
F1_train, F1_test = pd.read_csv(DATA_DIR / "F1_train.csv")[["statement", "label"]], pd.read_csv(DATA_DIR / "F1_test.csv")[["statement", "label"]]
F2_train, F2_test = pd.read_csv(DATA_DIR / "F2_train.csv"), pd.read_csv(DATA_DIR / "F2_test.csv")
F3_train, F3_test = pd.read_csv(DATA_DIR / "F3_train.csv"), pd.read_csv(DATA_DIR / "F3_test.csv")
F4_train, F4_test = pd.read_csv(DATA_DIR / "F4_train.csv"), pd.read_csv(DATA_DIR / "F4_test.csv")
F5_train, F5_test = pd.read_csv(DATA_DIR / "F5_train.csv"), pd.read_csv(DATA_DIR / "F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv(DATA_DIR / "A1_train.csv"), pd.read_csv(DATA_DIR / "A1_test.csv")
A2_train, A2_test = pd.read_csv(DATA_DIR / "A2_train.csv"), pd.read_csv(DATA_DIR / "A2_test.csv")
A3_train, A3_test = pd.read_csv(DATA_DIR / "A3_train.csv"), pd.read_csv(DATA_DIR / "A3_test.csv")


In [3]:
datasets = {
    "F0_train": F0_train, "F0_test": F0_test,
    "F1_train": F1_train, "F1_test": F1_test,
    "F2_train": F2_train, "F2_test": F2_test,
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
    "A1_train": A1_train, "A1_test": A1_test,
    "A2_train": A2_train, "A2_test": A2_test,
    "A3_train": A3_train, "A3_test": A3_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F0_train: 1194
F0_test: 512
F1_train: 1194
F1_test: 512
F2_train: 1194
F2_test: 512
F3_train: 1398
F3_test: 600
F4_train: 1394
F4_test: 598
F5_train: 1383
F5_test: 593
A1_train: 700
A1_test: 300
A2_train: 700
A2_test: 300
A3_train: 700
A3_test: 300


In [4]:
import torch
from tqdm import tqdm

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def generate_activations(model, statements, layer, with_chat_template=True, batch_size=16):
    statements = list(statements)
    final_token_activations = []

    for i in range(0, len(statements), batch_size):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        final_token_activation = outputs.hidden_states[layer][:, -1, :]
        final_token_activations.append(final_token_activation.cpu())

    return torch.cat(final_token_activations, dim=0)

In [5]:
for dataset in tqdm([F0_test, F0_train,
                     F1_train, F1_test,
                     F2_train, F2_test,
                     F3_train, F3_test,
                     F4_test, F4_train,
                     F5_train, F5_test, 
                     A1_test, A1_train,
                     A2_test, A2_train, 
                     A3_test, A3_train]):
    # Lowered batch size to 8 (from 16) to prevent Out-Of-Memory (OOM) on 9B models
    activations = generate_activations(model, dataset["statement"], MIDDLE_LAYER, with_chat_template=True, batch_size=8)
    dataset["activations_chat"] = list(activations)
    activations = generate_activations(model, dataset["statement"], MIDDLE_LAYER, with_chat_template=False, batch_size=8)
    dataset["activations"] = list(activations)

100%|██████████| 18/18 [47:37<00:00, 158.74s/it]


In [6]:
F0_test["activations"]

0      [tensor(0.7266, dtype=torch.bfloat16), tensor(...
1      [tensor(0.1904, dtype=torch.bfloat16), tensor(...
2      [tensor(0.8516, dtype=torch.bfloat16), tensor(...
3      [tensor(0.6211, dtype=torch.bfloat16), tensor(...
4      [tensor(0.1260, dtype=torch.bfloat16), tensor(...
                             ...                        
507    [tensor(0.0957, dtype=torch.bfloat16), tensor(...
508    [tensor(0.6211, dtype=torch.bfloat16), tensor(...
509    [tensor(0.7344, dtype=torch.bfloat16), tensor(...
510    [tensor(0.6875, dtype=torch.bfloat16), tensor(...
511    [tensor(0.4824, dtype=torch.bfloat16), tensor(...
Name: activations, Length: 512, dtype: object

### Training the Model and Extracting AUROC Scores

In [7]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def train_probe_pytorch(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels

    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()


    # Mean-center using only the training mean
    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]

    # THIS is the entire "model": one linear layer, no bias.
    # w(x) = w^T x, no offset term -> passes through the origin.
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()  # sigmoid + binary cross-entropy, combined for numerical stability

    for step in range(1000):
        optimizer.zero_grad()
        logits = probe(X_train_t).squeeze(-1)   # w^T x for every example
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

    # Evaluate
    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()
    auroc = roc_auc_score(y_test, test_logits)

    return probe, train_mean, auroc

In [8]:
train_probe_pytorch((F0_train["activations"], F0_test["activations"]), (F0_train["label"], F0_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.7328035 ,  0.03871757, -0.21753591, ...,  0.01455147,
        -0.4906241 , -0.58819205], dtype=float32),
 0.792205810546875)

In [9]:
train_probe_pytorch((F1_train["activations"], F1_test["activations"]), (F1_train["label"], F1_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.7737831 , -0.46991426, -0.26508164, ...,  0.03752278,
        -0.66760415, -0.13821067], dtype=float32),
 0.790863037109375)

In [10]:
train_probe_pytorch((F2_train["activations"], F2_test["activations"]), (F2_train["label"], F2_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.9037711 , -1.3932799 ,  0.09746245, ...,  0.21143356,
         0.17579399,  0.29478619], dtype=float32),
 0.688720703125)

In [11]:
train_probe_pytorch((F3_train["activations"], F3_test["activations"]), (F3_train["label"], F3_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 1.3910315 , -0.85145795,  0.08415437, ..., -0.08797967,
        -0.17188762,  0.3654849 ], dtype=float32),
 0.5986777777777779)

In [12]:
train_probe_pytorch((F4_train["activations"], F4_test["activations"]), (F4_train["label"], F4_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 1.9089535 , -1.0787055 ,  0.22137453, ..., -0.36830664,
        -0.22161826,  0.8806178 ], dtype=float32),
 0.5463920985223878)

In [13]:
train_probe_pytorch((F5_train["activations"], F5_test["activations"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 2.2364087 , -0.95478094, -0.5115185 , ..., -0.92026895,
        -0.11547276,  0.94319284], dtype=float32),
 0.5564883064883065)

In [14]:
train_probe_pytorch((A1_train["activations"], A1_test["activations"]), (A1_train["label"], A1_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([-0.26790985, -0.19859196, -0.04334504, ..., -0.28655753,
        -0.2640724 , -0.30792803], dtype=float32),
 0.3099111111111111)

In [15]:
train_probe_pytorch((A2_train["activations"], A2_test["activations"]), (A2_train["label"], A2_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([-0.16106239, -0.42931482, -0.17651506, ..., -0.25118887,
        -0.32321522, -0.02057757], dtype=float32),
 0.3388444444444444)

In [16]:
train_probe_pytorch((A3_train["activations"], A3_test["activations"]), (A3_train["label"], A3_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([-0.28853327, -0.31433955, -0.05587741, ..., -0.07786132,
        -0.33111522, -0.05198979], dtype=float32),
 0.2699111111111111)

In [17]:
train_probe_pytorch((F0_train["activations_chat"], F0_test["activations_chat"]), (F0_train["label"], F0_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([-0.70177156,  1.1097846 , -0.5173137 , ..., -1.314638  ,
        -0.5818703 ,  1.186424  ], dtype=float32),
 0.9932708740234375)

In [18]:
train_probe_pytorch((F1_train["activations_chat"], F1_test["activations_chat"]), (F1_train["label"], F1_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.10154976,  1.046604  ,  0.7474342 , ..., -0.13496381,
        -0.06739218,  1.9135653 ], dtype=float32),
 0.997222900390625)

In [19]:
train_probe_pytorch((F2_train["activations_chat"], F2_test["activations_chat"]), (F2_train["label"], F2_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.18040623,  0.16545238,  0.38594553, ..., -0.8789995 ,
        -0.60385424,  1.1957018 ], dtype=float32),
 0.9772491455078125)

In [20]:
train_probe_pytorch((F3_train["activations_chat"], F3_test["activations_chat"]), (F3_train["label"], F3_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.2313318 ,  0.36482856,  1.0706925 , ..., -1.2238382 ,
        -0.62519026,  1.8261299 ], dtype=float32),
 0.9336166666666665)

In [21]:
train_probe_pytorch((F4_train["activations_chat"], F4_test["activations_chat"]), (F4_train["label"], F4_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 1.0780058 ,  0.7299149 ,  1.2425812 , ..., -0.9558703 ,
        -0.78588134,  2.1251402 ], dtype=float32),
 0.88964329258062)

In [22]:
train_probe_pytorch((F5_train["activations_chat"], F5_test["activations_chat"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.25374758,  1.5094733 ,  0.8725527 , ..., -0.66985196,
        -0.7270695 ,  1.6857601 ], dtype=float32),
 0.8148261898261898)

In [23]:
train_probe_pytorch((A1_train["activations_chat"], A1_test["activations_chat"]), (A1_train["label"], A1_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([ 0.19634421,  0.21685837,  0.31114584, ..., -0.775897  ,
        -0.28875366,  0.7441841 ], dtype=float32),
 0.6383555555555556)

In [24]:
train_probe_pytorch((A2_train["activations_chat"], A2_test["activations_chat"]), (A2_train["label"], A2_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([-0.32246342,  0.6376831 ,  0.38080344, ..., -1.0910652 ,
        -0.759112  ,  0.43653303], dtype=float32),
 0.6265777777777777)

In [25]:
train_probe_pytorch((A3_train["activations_chat"], A3_test["activations_chat"]), (A3_train["label"], A3_test["label"]))

(Linear(in_features=1536, out_features=1, bias=False),
 array([-0.77901   ,  0.9189655 ,  0.4078357 , ..., -0.83820105,
        -0.8290625 ,  0.30008817], dtype=float32),
 0.4912888888888889)

*Note: This notebook is generalized to run on the latest small/edge models (e.g., Gemma 4 E2B, Phi-4-mini, Qwen3). Results may vary across model families depending on chat-template sensitivity.*


We find that chat-template application is **critical** for the Gemma 4 Edge model, dramatically improving linear separability across all tasks. Without the chat template, performance degrades severely (e.g., F0 drops from 0.993 to 0.792) and arithmetic tasks drop below chance (A3 drops from 0.491 to 0.270). This is in stark contrast to Llama-3.1-8B-Instruct, highlighting that some instruction-tuned models are highly sensitive to chat-formatting for their internal representations. Consequently, we must adopt chat-templates as our primary methodology when extracting truth directions from this model family.